# 04 — Scoring Validation

Sanity check priority rankings and compare against baselines.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from featrank.ingest.csv_connector import CSVConnector
from featrank.pipeline.cleaner import TextCleaner
from featrank.pipeline.embedder import Embedder
from featrank.pipeline.clusterer import HDBSCANClusterer
from featrank.pipeline.labeler import ClusterLabeler
from featrank.scoring.ranker import CompositeRanker

sns.set_theme(style='whitegrid')

In [ ]:
requests = CSVConnector('../tests/fixtures/sample_requests.csv').fetch_all()
cleaner = TextCleaner()
kept, cleaned_texts = cleaner.clean_requests(requests)
embedder = Embedder(model_name='all-MiniLM-L6-v2', cache_dir='.cache/minilm')
embeddings = embedder.embed(cleaned_texts)
clusterer = HDBSCANClusterer(min_cluster_size=5, min_samples=3)
clustered = clusterer.fit_predict(kept, embeddings, cleaned_texts)
labeler = ClusterLabeler()
clusters = labeler.build_clusters(clustered)
print(f'{len(clusters)} clusters built')

In [ ]:
roadmap = open('../tests/fixtures/sample_roadmap.txt').read()
ranker = CompositeRanker(github_repo='')
ranked = ranker.rank(clusters, roadmap_text=roadmap)

df = pd.DataFrame([{
    'rank': c.priority_rank,
    'label': c.label,
    'score': c.priority_score,
    'freq': c.score_frequency,
    'seg': c.score_segment_value,
    'gh': c.score_github_signal,
    'road': c.score_roadmap_fit,
    'requests': c.request_count,
    'mrr': c.total_mrr,
} for c in ranked])

df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top10 = df.head(10)
axes[0].barh(top10['label'][::-1], top10['score'][::-1])
axes[0].set_xlabel('Priority Score (0-100)')
axes[0].set_title('Top 10 Feature Clusters by Priority Score')

signal_cols = ['freq', 'seg', 'gh', 'road']
top10[signal_cols].plot(kind='bar', ax=axes[1], stacked=False)
axes[1].set_xticklabels(top10['label'], rotation=30, ha='right', fontsize=7)
axes[1].set_title('Signal Breakdown — Top 10 Clusters')
axes[1].legend(['Frequency', 'Segment Value', 'GitHub', 'Roadmap Fit'])

plt.tight_layout()
plt.show()

In [ ]:
import math, random

def ndcg(ranked_ids, ideal_ids, relevance, k):
    def dcg(ids):
        return sum(relevance.get(i, 0) / math.log2(pos + 2) for pos, i in enumerate(ids[:k]))
    return dcg(ranked_ids) / dcg(sorted(ideal_ids, key=lambda i: relevance.get(i, 0), reverse=True)) or 0

max_mrr = max(c.total_mrr for c in ranked) or 1
max_cnt = max(c.request_count for c in ranked) or 1
relevance = {c.cluster_id: round((0.6*(c.total_mrr/max_mrr) + 0.4*(c.request_count/max_cnt))*3)
             for c in ranked}
all_ids = [c.cluster_id for c in ranked]

featrank_ids = [c.cluster_id for c in ranked]
freq_ids = [c.cluster_id for c in sorted(ranked, key=lambda c: c.request_count, reverse=True)]
rand_ids = all_ids[:]; random.Random(42).shuffle(rand_ids)

results = []
for name, ids in [('Random', rand_ids), ('Frequency-only', freq_ids), ('featrank', featrank_ids)]:
    results.append({'Method': name,
                    'NDCG@5': round(ndcg(ids, all_ids, relevance, 5), 3),
                    'NDCG@10': round(ndcg(ids, all_ids, relevance, 10), 3)})

pd.DataFrame(results)